# Step 1 — Fine-tuned Baselines: BERT, HateBERT, and RoBERTa on IHC and ISHate

We fine-tune three pretrained models on two hate speech corpora (IHC and ISHate) and compare their macro F1 scores across both datasets.

In [1]:
# Uncomment if running in Colab
# !pip install -q transformers datasets accelerate scikit-learn

In [2]:
import numpy as np
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
import warnings
warnings.filterwarnings("ignore")

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load the Datasets

### IHC (Implicit Hate Corpus)
`SALT-NLP/implicit-hate` was removed from the Hub — `tasksource/implicit-hate-stg1` is a verified mirror of the same IHC corpus (ElSherief et al., 2021). It has a single `train` split with 21,480 tweets labelled as `not_hate`, `explicit_hate`, or `implicit_hate`. Since there is no pre-made test split, we hold out 10% for evaluation. The binary label collapses explicit and implicit hate into a single **HS** class (label = 1); non-hate is **Non-HS** (label = 0).

### ISHate (Implicit and Subtle Hate Speech)
`BenjaminOcampo/ISHate` is a large-scale dataset (63,758 examples) focused on implicit and subtle hate speech, annotated with multiple layers. We use the `hateful_layer` column for binary classification (`Non-HS` → 0, `HS` → 1) and its pre-made train/test splits.

In [3]:
# IHC
raw = load_dataset("tasksource/implicit-hate-stg1", split="train")
splits = raw.train_test_split(test_size=0.10, seed=42)

def add_binary_label_ihc(example):
    example["label"] = 0 if example["class"] == "not_hate" else 1
    return example

train_ds = splits["train"].map(add_binary_label_ihc)
test_ds  = splits["test"].map(add_binary_label_ihc)

print("IHC")
print(f"  Train: {len(train_ds):,}  Test: {len(test_ds):,}")

# ISHate
ishate_raw = load_dataset("BenjaminOcampo/ISHate")

def add_binary_label_ishate(example):
    example["label"] = 0 if example["hateful_layer"] == "Non-HS" else 1
    return example

ishate_train = ishate_raw["train"].map(add_binary_label_ishate)
ishate_test  = ishate_raw["test"].map(add_binary_label_ishate)

print("\nISHate")
print(f"  Train: {len(ishate_train):,}  Test: {len(ishate_test):,}")

Generating train split:   0%|          | 0/21480 [00:00<?, ? examples/s]

Generating train split: 100%|██████████| 21480/21480 [00:00<00:00, 512119.79 examples/s]

Map:   0%|          | 0/19332 [00:00<?, ? examples/s]

Map:  12%|█▏        | 2272/19332 [00:00<00:00, 22605.28 examples/s]

Map:  28%|██▊       | 5451/19332 [00:00<00:00, 21638.89 examples/s]

...

Map:   0%|          | 0/4368 [00:00<?, ? examples/s]

Map:  46%|████▌     | 2000/4368 [00:00<00:00, 19580.75 examples/s]

Map:  99%|█████████▉| 4318/4368 [00:00<00:00, 10141.09 examples/s]

Map: 100%|██████████| 4368/4368 [00:00<00:00, 10666.51 examples/s]


ISHate
  Train: 55,023  Test: 4,368


## 2. Tokenization

Transformer models cannot work directly on raw text — they require a **tokenizer** that converts text into integer token IDs and an attention mask.

We truncate and pad every sequence to a fixed length of 128 tokens. Most tweets are well under this limit, so little information is lost. Each model has its own vocabulary and tokenizer, so we re-tokenize for each model.

In [4]:
def tokenize(ds, tokenizer, text_col="post", max_length=128):
    def _tok(batch):
        return tokenizer(
            batch[text_col],
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )
    return ds.map(_tok, batched=True)

## 3. Metrics

The `Trainer` calls `compute_metrics` after each evaluation epoch. We use **macro F1** as the primary metric — it averages F1 across both classes, penalising models that ignore the minority class (hate speech). Precision and recall give additional diagnostic information.

In [5]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "macro_f1":  f1_score(labels, preds, average="macro",  zero_division=0),
        "macro_p":   precision_score(labels, preds, average="macro", zero_division=0),
        "macro_r":   recall_score(labels, preds, average="macro",    zero_division=0),
    }

## 4. Fine-tuning Loop

### What is a classification head?

A pretrained model like BERT has learned rich representations of text but has no built-in notion of hate speech. `AutoModelForSequenceClassification` adds a small **linear layer** (the classification head) on top of the `[CLS]` token representation, projecting it to 2 output logits (Non-HS, HS). This head is randomly initialised.

### What does fine-tuning do?

Fine-tuning runs gradient descent on the labelled IHC training set, updating **all** model weights — both the pretrained transformer layers and the new classification head. The pretrained weights are a warm start: the model has already learned grammar, semantics, and world knowledge, so it only needs a few epochs to adapt to the new task. Training from scratch would require far more data and compute.

In [6]:
MODELS = {
    "bert-base-uncased": "bert-base-uncased",
    "hateBERT":          "GroNLP/hateBERT",
    "roberta-base":      "roberta-base",
}

DATASETS = {
    "IHC":    {"train": train_ds,    "test": test_ds,    "text_col": "post"},
    "ISHate": {"train": ishate_train, "test": ishate_test, "text_col": "text"},
}

results = {}

for dataset_name, dataset in DATASETS.items():
    results[dataset_name] = {}
    for model_name, model_id in MODELS.items():
        print(f"\n{'='*60}")
        print(f"Dataset: {dataset_name}  |  Model: {model_name}")
        print(f"{'='*60}")

        tokenizer = AutoTokenizer.from_pretrained(model_id)
        model     = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)

        tok_train = tokenize(dataset["train"], tokenizer, text_col=dataset["text_col"])
        tok_test  = tokenize(dataset["test"],  tokenizer, text_col=dataset["text_col"])

        training_args = TrainingArguments(
            output_dir=f"./checkpoints/{dataset_name}/{model_name}",
            num_train_epochs=3,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=32,
            learning_rate=2e-5,
            eval_strategy="epoch",
            save_strategy="no",
            logging_strategy="epoch",
            report_to="none",
            seed=42,
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=tok_train,
            eval_dataset=tok_test,
            compute_metrics=compute_metrics,
        )

        trainer.train()

        preds_output = trainer.predict(tok_test)
        preds  = np.argmax(preds_output.predictions, axis=-1)
        labels = dataset["test"]["label"]

        print(classification_report(labels, preds, target_names=["Non-HS", "HS"]))

        results[dataset_name][model_name] = {
            "macro_f1":  f1_score(labels, preds, average="macro",  zero_division=0),
            "macro_p":   precision_score(labels, preds, average="macro", zero_division=0),
            "macro_r":   recall_score(labels, preds, average="macro",    zero_division=0),
        }


Dataset: IHC  |  Model: bert-base-uncased


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7577.34it/s]


BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/19332 [00:00<?, ? examples/s]

...

Map:  46%|████▌     | 2000/4368 [00:00<00:00, 15995.12 examples/s]

Map: 100%|██████████| 4368/4368 [00:00<00:00, 18105.93 examples/s]

Map: 100%|██████████| 4368/4368 [00:00<00:00, 17478.53 examples/s]

Epoch,Training Loss,Validation Loss,Macro F1,Macro P,Macro R
1,0.174640,0.403467,0.856421,0.879242,0.844940
2,0.116572,0.387130,0.882047,0.880292,0.884035
3,0.081452,0.477076,0.885960,0.885426,0.886513


              precision    recall  f1-score   support

      Non-HS       0.91      0.91      0.91      2681
          HS       0.86      0.86      0.86      1687

    accuracy                           0.89      4368
   macro avg       0.89      0.89      0.89      4368
weighted avg       0.89      0.89      0.89      4368



## 5. Results

One table per dataset shows macro F1/P/R for all three models. Comparing across datasets reveals whether model rankings are consistent or dataset-dependent — e.g., whether HateBERT's domain-specific pretraining helps more on ISHate's subtler examples than on IHC.

In [7]:
import pandas as pd

for dataset_name, dataset_results in results.items():
    df = pd.DataFrame(dataset_results).T
    df.columns = ["Macro F1", "Macro Precision", "Macro Recall"]
    df.index.name = "Model"
    styled = df.style.format("{:.3f}").highlight_max(axis=0, props="font-weight: bold; background-color: #d4f1d4").set_caption(dataset_name)
    display(styled)

,Macro F1,Macro Precision,Macro Recall
Model,,,
bert-base-uncased,0.790,0.795,0.787
hateBERT,0.783,0.790,0.778
roberta-base,0.793,0.799,0.789


,Macro F1,Macro Precision,Macro Recall
Model,,,
bert-base-uncased,0.871,0.871,0.872
hateBERT,0.876,0.876,0.875
roberta-base,0.886,0.885,0.887
